## Group-level RSA (CWS vs. CWNS) on GLMsingle single-trial ROI RDMs

Compares CWS and CWNS on the per-subject, per-ROI crossnobis RDMs computed by `GLMsingle_rsa-roi.py`. Two complementary analyses:
1. **Model-fit comparison** (primary): correlate each subject's empirical RDM (per ROI) against categorical model RDMs (SNR level, syllable identity, speaker identity), then compare model-fit strength between groups via subject-level bootstrap.
2. **Direct RDM comparison** (descriptive): compare CWS-mean vs. CWNS-mean RDM per ROI directly, no categorical models involved.

**Multiple comparisons note (deliberate choice):** every ROI x model test below is independent, uncorrected across the 20 ROIs x up to 3 models -- consistent with the same "keep independent per-contrast FDR, document as deliberate" choice already made in `univariate_group-level.ipynb`.

In [ ]:
import os
from glob import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import rsatoolbox

### Set parameters

In [ ]:
bidsroot = os.path.join('/bgfs/bchandrasekaran/krs228/data/',
                        'SSP/',
                        'data_bids')
glmsingle_dir = os.path.join(bidsroot, 'derivatives', 'glmsingle')

RDM_METHOD = 'crossnobis'
NOISE_LEVEL_TAG = 'all'  # matches GLMsingle_rsa-roi.py's --noise_level output tag ('all' or e.g. 'Q')

rdm_dir = os.path.join(glmsingle_dir, f'rsa-roi_glmsingle_rdmcalc-{RDM_METHOD}')
print('rdm_dir:', rdm_dir)

CORTICAL_ROI_LIST = [
    'L-HG', 'L-PT', 'L-PP', 'L-STGp', 'L-STGa', 'L-ParsOp', 'L-ParsTri',
    'R-HG', 'R-PT', 'R-PP', 'R-STGp', 'R-STGa', 'R-ParsOp', 'R-ParsTri',
    'L-SMGa', 'L-SMGp', 'L-Ang',
    'R-SMGa', 'R-SMGp', 'R-Ang',
]

N_BOOT = 10000
RNG = np.random.default_rng(0)

out_dir = os.path.join(glmsingle_dir, 'rsa-group_glmsingle')
os.makedirs(out_dir, exist_ok=True)

### Load participants and group membership

In [ ]:
participants_fpath = os.path.join(bidsroot, 'participants.tsv')
participants_df = pd.read_csv(participants_fpath, sep='\t')

# case-/whitespace-normalized group comparison -- same fix already applied in
# univariate_group-level.ipynb, since participants.tsv's group column has been observed with
# inconsistent casing (e.g. 'CWS' vs 'cws').
group_norm = participants_df.group.str.strip().str.lower()
sub_list_cwns_all = list(participants_df.participant_id[group_norm == 'control'])
sub_list_cws_all = list(participants_df.participant_id[group_norm == 'cws'])

# restrict to subjects that actually have an RDM file -- not every subject in participants.tsv
# has usable GLMsingle output (e.g. single-run subjects only get the degraded TYPEB estimate,
# which GLMsingle_mask-betas.py already refuses to mask for RSA).
rdm_fpaths = sorted(glob(os.path.join(rdm_dir, f'sub-*_glmsingle_cortical_{RDM_METHOD}_noiselevel-{NOISE_LEVEL_TAG}_rdms.hdf5')))
subs_with_rdms = set()
for f in rdm_fpaths:
    fname = os.path.basename(f)
    sub_id = fname.split('_')[0].replace('sub-', '')
    subs_with_rdms.add(f'sub-{sub_id}')

sub_list_cwns = [s for s in sub_list_cwns_all if s in subs_with_rdms]
sub_list_cws = [s for s in sub_list_cws_all if s in subs_with_rdms]

print(f'{len(rdm_fpaths)} RDM files found')
print(f'CWNS: {len(sub_list_cwns)}/{len(sub_list_cwns_all)} have RDMs')
print(f'CWS: {len(sub_list_cws)}/{len(sub_list_cws_all)} have RDMs')

### Load each subject's RDMs

In [ ]:
def load_subject_rdms(sub_id):
    fpath = os.path.join(rdm_dir, f'{sub_id}_glmsingle_cortical_{RDM_METHOD}_noiselevel-{NOISE_LEVEL_TAG}_rdms.hdf5')
    # NOTE: rsatoolbox's exact load function/path is not independently confirmed against an
    # installed version (none available locally) -- rsatoolbox.rdm.rdms.load_rdm is the
    # symmetric counterpart to rsatoolbox.rdm.rdms.concat, which GLMsingle_rsa-roi.py already
    # uses successfully to save these files. Confirm this exact call on first real run; if it
    # differs, this is the one line to fix.
    return rsatoolbox.rdm.rdms.load_rdm(fpath, file_type='hdf5')


all_subs = sub_list_cwns + sub_list_cws
subject_rdms = {}
for sub_id in all_subs:
    subject_rdms[sub_id] = load_subject_rdms(sub_id)

print(f'Loaded RDMs for {len(subject_rdms)} subjects')

### Categorical model RDMs (SNR / syllable / speaker)

In [ ]:
def build_categorical_model_rdms(pattern_labels):
    """Build SNR/syllable/speaker categorical model RDMs (0 if the two stimuli match on that
    dimension, 1 if they differ), sized and ORDERED to exactly match `pattern_labels` -- the
    actual condition order taken from a loaded empirical RDM's pattern_descriptors, not a
    freshly re-derived list. The model and empirical RDMs' row/column order must match exactly
    for rsatoolbox.rdm.compare() to compare like-for-like conditions.

    Each condition label is expected to be 'SYLLABLE_SPEAKER_NOISELEVEL' (e.g. 'BA_F1_Q'), per
    GLMsingle_first-level.py's build_condition_labels(). Fixes the copy-paste bug in the old
    group_level_rsa_searchlight_WIP.ipynb, where the talker/speaker model RDM was assigned into
    the syllable_rdms variable instead of its own.
    """
    n = len(pattern_labels)
    parsed = [label.split('_') for label in pattern_labels]
    for p in parsed:
        assert len(p) == 3, f"expected 'syllable_speaker_noiselevel', got {p!r}"
    syllables = [p[0] for p in parsed]
    speakers = [p[1] for p in parsed]
    noise_levels = [p[2] for p in parsed]

    model_rdms = {}
    model_rdms['syllable'] = np.array([[0 if syllables[i] == syllables[j] else 1
                                        for j in range(n)] for i in range(n)])
    model_rdms['speaker'] = np.array([[0 if speakers[i] == speakers[j] else 1
                                       for j in range(n)] for i in range(n)])

    # SNR model is degenerate (all-zero, no variance to model) if every trial shares the same
    # noise level -- e.g. a noiselevel-restricted (Q-only) RDM. Skip it automatically rather
    # than including a meaningless all-zero model, instead of hardcoding per noise-level-tag.
    if len(set(noise_levels)) > 1:
        model_rdms['snr'] = np.array([[0 if noise_levels[i] == noise_levels[j] else 1
                                       for j in range(n)] for i in range(n)])
    else:
        print(f'Only one noise level ({noise_levels[0]}) present -- skipping the degenerate SNR model.')

    return model_rdms


# pattern order is assumed identical across subjects/ROIs (same fixed condition design for
# everyone) -- take it from the first subject's first ROI as the reference.
example_rdms = next(iter(subject_rdms.values()))
example_roi_rdm = example_rdms.subset('ROI', CORTICAL_ROI_LIST[0])
# NOTE: pattern_descriptors key name ('stimulus') matches the `descriptor='stimulus'` used when
# building these RDMs in GLMsingle_rsa-roi.py's calc_rdm() call -- confirm on first real run.
pattern_labels = list(example_roi_rdm.pattern_descriptors['stimulus'])
print(f'{len(pattern_labels)} conditions per RDM:', pattern_labels[:5], '...')

categorical_model_rdms = build_categorical_model_rdms(pattern_labels)
print('models built:', list(categorical_model_rdms.keys()))

#### QC: visualize the categorical model RDMs

In [ ]:
fig, axes = plt.subplots(1, len(categorical_model_rdms), figsize=(5 * len(categorical_model_rdms), 4), dpi=150)
if len(categorical_model_rdms) == 1:
    axes = [axes]
for ax, (name, mat) in zip(axes, categorical_model_rdms.items()):
    sns.heatmap(mat, cmap='Greys', square=True, cbar=False, xticklabels=False, yticklabels=False, ax=ax)
    ax.set_title(name)
fig.tight_layout()

### Per-subject, per-ROI model-fit scalars

In [ ]:
def get_roi_rdm_vector(rdms_obj, roi):
    """Extract the single dissimilarity vector for one ROI from a subject's full RDMs object.
    NOTE: .get_vectors() is rsatoolbox's compressed (upper-triangular) vector form -- confirm
    this exact method name on first real run; .get_matrices() is the square-matrix alternative
    if this doesn't match the installed rsatoolbox version.
    """
    roi_rdm = rdms_obj.subset('ROI', roi)
    return roi_rdm.get_vectors()[0]


model_fit_rows = []
for sub_id, rdms_obj in subject_rdms.items():
    group = 'CWNS' if sub_id in sub_list_cwns else 'CWS'
    for roi in CORTICAL_ROI_LIST:
        try:
            roi_rdm = rdms_obj.subset('ROI', roi)
        except Exception as e:
            print(f'Could not extract ROI {roi} for {sub_id}: {e}')
            continue

        for model_name, model_mat in categorical_model_rdms.items():
            model_rdm_obj = rsatoolbox.rdm.RDMs(
                model_mat[np.newaxis, :, :],
                rdm_descriptors={'model': [model_name]},
                dissimilarity_measure='Euclidean',
            )
            # NOTE: rsatoolbox.rdm.compare()'s exact signature/return shape (scalar vs. 1x1
            # array) is not independently confirmed against an installed version -- adjust the
            # indexing below if this doesn't match on first real run.
            fit = rsatoolbox.rdm.compare(roi_rdm, model_rdm_obj, method='corr')
            fit_value = np.asarray(fit).flatten()[0]

            model_fit_rows.append({
                'subject_id': sub_id, 'group': group, 'ROI': roi,
                'model': model_name, 'fit': fit_value,
            })

model_fit_df = pd.DataFrame(model_fit_rows)
model_fit_df.to_csv(os.path.join(out_dir, f'model_fit_scalars_noiselevel-{NOISE_LEVEL_TAG}.csv'), index=False)
model_fit_df.head()

### Group comparison: subject-level bootstrap

In [ ]:
def bootstrap_group_difference(values_a, values_b, n_boot=N_BOOT, rng=RNG):
    """Subject-level bootstrap test for a difference in means between two groups. Resamples
    subjects WITH replacement WITHIN each group (not across groups), computes the group-mean
    difference (a - b) each iteration, and derives a two-sided empirical p-value from how often
    the null-centered bootstrap distribution is at least as extreme as the observed difference.
    Operates purely on whatever scalars are passed in -- agnostic to how they were derived,
    including from GLMsingle-based RDMs (confirmed compatible per the earlier discussion: the
    bootstrap is downstream of and independent from how the single-trial patterns were estimated).
    """
    values_a = np.asarray(values_a)
    values_b = np.asarray(values_b)
    observed_diff = values_a.mean() - values_b.mean()

    boot_diffs = np.empty(n_boot)
    for i in range(n_boot):
        resampled_a = rng.choice(values_a, size=len(values_a), replace=True)
        resampled_b = rng.choice(values_b, size=len(values_b), replace=True)
        boot_diffs[i] = resampled_a.mean() - resampled_b.mean()

    ci_low, ci_high = np.percentile(boot_diffs, [2.5, 97.5])
    null_centered = boot_diffs - boot_diffs.mean()
    p_value = np.mean(np.abs(null_centered) >= np.abs(observed_diff))

    return {'observed_diff': observed_diff, 'ci_low': ci_low, 'ci_high': ci_high, 'p_value': p_value}


group_comparison_rows = []
for roi in CORTICAL_ROI_LIST:
    for model_name in categorical_model_rdms.keys():
        subset = model_fit_df[(model_fit_df.ROI == roi) & (model_fit_df.model == model_name)]
        cws_vals = subset.loc[subset.group == 'CWS', 'fit'].values
        cwns_vals = subset.loc[subset.group == 'CWNS', 'fit'].values
        if len(cws_vals) < 2 or len(cwns_vals) < 2:
            print(f'Skipping {roi}/{model_name}: not enough subjects per group')
            continue

        result = bootstrap_group_difference(cws_vals, cwns_vals)
        group_comparison_rows.append({
            'ROI': roi, 'model': model_name,
            'n_cws': len(cws_vals), 'n_cwns': len(cwns_vals),
            **result,
        })

group_comparison_df = pd.DataFrame(group_comparison_rows)
group_comparison_df.to_csv(os.path.join(out_dir, f'group_comparison_bootstrap_noiselevel-{NOISE_LEVEL_TAG}.csv'), index=False)
group_comparison_df.sort_values('p_value').head(20)

### Direct RDM comparison (descriptive complement)

In [ ]:
def bootstrap_rdm_distance(rdm_vectors_a, rdm_vectors_b, n_boot=N_BOOT, rng=RNG):
    """Bootstrap CI for whether two groups' mean RDMs differ, using 1 - Pearson correlation
    between the two group-mean RDM vectors as the dissimilarity-of-dissimilarities statistic.
    Descriptive complement to the categorical-model-fit comparison above -- no categorical
    models involved, just "is the overall similarity structure different between groups."
    Doesn't have a natural null-hypothesis p-value the way a mean-difference bootstrap does;
    report the CI and whether it's close to 0 (little difference) or not.
    """
    rdm_vectors_a = np.vstack(rdm_vectors_a)
    rdm_vectors_b = np.vstack(rdm_vectors_b)

    def rdm_corr_distance(vecs_a, vecs_b):
        mean_a = vecs_a.mean(axis=0)
        mean_b = vecs_b.mean(axis=0)
        r = np.corrcoef(mean_a, mean_b)[0, 1]
        return 1 - r

    observed_distance = rdm_corr_distance(rdm_vectors_a, rdm_vectors_b)

    idx_a = np.arange(len(rdm_vectors_a))
    idx_b = np.arange(len(rdm_vectors_b))
    boot_distances = np.empty(n_boot)
    for i in range(n_boot):
        resampled_a = rdm_vectors_a[rng.choice(idx_a, size=len(idx_a), replace=True)]
        resampled_b = rdm_vectors_b[rng.choice(idx_b, size=len(idx_b), replace=True)]
        boot_distances[i] = rdm_corr_distance(resampled_a, resampled_b)

    ci_low, ci_high = np.percentile(boot_distances, [2.5, 97.5])
    return {'observed_distance': observed_distance, 'ci_low': ci_low, 'ci_high': ci_high}


mean_rdm_rows = []
for roi in CORTICAL_ROI_LIST:
    cws_vectors = [get_roi_rdm_vector(subject_rdms[s], roi) for s in sub_list_cws if s in subject_rdms]
    cwns_vectors = [get_roi_rdm_vector(subject_rdms[s], roi) for s in sub_list_cwns if s in subject_rdms]
    if len(cws_vectors) < 2 or len(cwns_vectors) < 2:
        continue

    result = bootstrap_rdm_distance(cws_vectors, cwns_vectors)
    mean_rdm_rows.append({'ROI': roi, **result})

mean_rdm_df = pd.DataFrame(mean_rdm_rows)
mean_rdm_df.to_csv(os.path.join(out_dir, f'mean_rdm_comparison_noiselevel-{NOISE_LEVEL_TAG}.csv'), index=False)
mean_rdm_df.sort_values('observed_distance', ascending=False)

#### QC: visualize CWS-mean vs. CWNS-mean RDM for one ROI

In [ ]:
example_roi = CORTICAL_ROI_LIST[0]
cws_vectors = [get_roi_rdm_vector(subject_rdms[s], example_roi) for s in sub_list_cws if s in subject_rdms]
cwns_vectors = [get_roi_rdm_vector(subject_rdms[s], example_roi) for s in sub_list_cwns if s in subject_rdms]

fig, axes = plt.subplots(1, 2, figsize=(10, 4), dpi=150)
sns.heatmap(np.vstack(cws_vectors).mean(axis=0).reshape(1, -1), cmap='viridis', ax=axes[0], cbar=True)
axes[0].set_title(f'CWS mean RDM (vectorized) -- {example_roi}')
sns.heatmap(np.vstack(cwns_vectors).mean(axis=0).reshape(1, -1), cmap='viridis', ax=axes[1], cbar=True)
axes[1].set_title(f'CWNS mean RDM (vectorized) -- {example_roi}')
fig.tight_layout()

### Summary

In [ ]:
print(f'Model-fit scalars: {len(model_fit_df)} rows -> '
     f'{os.path.join(out_dir, f"model_fit_scalars_noiselevel-{NOISE_LEVEL_TAG}.csv")}')
print(f'Group comparison (bootstrap): {len(group_comparison_df)} ROI x model tests -> '
     f'{os.path.join(out_dir, f"group_comparison_bootstrap_noiselevel-{NOISE_LEVEL_TAG}.csv")}')
print(f'Direct mean-RDM comparison: {len(mean_rdm_df)} ROIs -> '
     f'{os.path.join(out_dir, f"mean_rdm_comparison_noiselevel-{NOISE_LEVEL_TAG}.csv")}')
print()
print('Uncorrected (deliberately -- see top of notebook) significant ROI x model tests, p < 0.05:')
print(group_comparison_df[group_comparison_df.p_value < 0.05].sort_values('p_value'))